<a href="https://colab.research.google.com/github/adi542/ai-ml/blob/main/Tranformer_Encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoTokenizer

In [ ]:
!pip install bertviz

In [ ]:
from bertviz.transformers_neuron_view import BertModel

In [ ]:
from bertviz.neuron_view import show


In [ ]:
model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = BertModel.from_pretrained(model_ckpt)
text = "time flies like an arrow"


In [ ]:
show(model,"bert",tokenizer,text,display_mode="dark",layer=0,head=8)

In [ ]:
inputs = tokenizer(text,return_tensors="pt",add_special_tokens=False)


In [ ]:
inputs.input_ids

In [ ]:
from torch import nn
from transformers import AutoConfig
config = AutoConfig.from_pretrained(model_ckpt)
token_emb = nn.Embedding(config.vocab_size,config.hidden_size)
token_emb

In [ ]:
inputs_emb = token_emb(inputs.input_ids)
inputs_emb.shape

In [ ]:
import torch
from math import sqrt

In [ ]:
query=key=value = inputs_emb
dim_k = key.size(-1)
scores = torch.bmm(query,key.transpose(1,2))/sqrt(dim_k)
scores.shape

In [ ]:
import torch.nn.functional as F
weights = F.softmax(scores,dim=-1)

In [ ]:
weights.sum(dim=-1)

In [ ]:
attn_outputs = torch.bmm(weights,value)
attn_outputs.shape

In [ ]:
def scaled_dot_product_attention(query,key,value):
  dim_k = query.size(-1)
  scores = torch.bmm(query,key.transpose(1,2))/sqrt(dim_k)
  weights = F.softmax(scores,dim=-1)
  return torch.bmm(weights,value)

In [ ]:
class AttentionHead(nn.Module):
  def __init__(self,emded_dim,head_dim):
    super().__init__()
    self.q = nn.Linear(emded_dim,head_dim)
    self.k = nn.Linear(emded_dim,head_dim)
    self.v = nn.Linear(emded_dim,head_dim)

  def forward(self,hidden_state):
    attn_outputs = scaled_dot_product_attention(
        self.q(hidden_state),self.k(hidden_state),self.v(hidden_state)
    )
    return attn_outputs



In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self,config):
    super().__init__()
    embed_dim = config.hidden_size
    num_heads = config.num_attention_heads
    head_dim = embed_dim//num_heads
    self.heads = nn.ModuleList(
        [AttentionHead(embed_dim,head_dim) for _ in range(num_heads)]
    )
    self.output_linear = nn.Linear(embed_dim,embed_dim)

  def forward(self,hidden_state):
    x = torch.cat([h(hidden_state) for h in self.heads],dim=-1)
    x = self.output_linear(x)
    return x

In [ ]:
config

In [ ]:
inputs_emb

In [ ]:
multihead_attn = MultiHeadAttention(config)
attn_output = multihead_attn(inputs_emb)
attn_output.size()

In [ ]:
class FeedForward(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.linear_1 = nn.Linear(config.hidden_size,config.intermediate_size)
    self.linear_2 = nn.Linear(config.intermediate_size,config.hidden_size)
    self.gelu = nn.GELU()
    self.dropout = nn.Dropout(config.hidden_dropout_prob)

  def forward(self,x):
    x = self.linear_1(x)
    x = self.gelu(x)
    x = self.linear_2(x)
    x = self.dropout(x)
    return x

In [ ]:
feed_forward = FeedForward(config)
ff_outputs = feed_forward(attn_output)
ff_outputs.size()

In [ ]:
class TransformerEncoderLayer(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.layer_norm_1 = nn.LayerNorm(config.hidden_size)
    self.layer_norm_2 = nn.LayerNorm(config.hidden_size)
    self.attention = MultiHeadAttention(config)
    self.feed_forward = FeedForward(config)
  def forward(self,hidden_state):
    x = hidden_state
    x = x + self.attention(self.layer_norm_1(x))
    x = x + self.feed_forward(self.layer_norm_2(x))
    return x

In [ ]:
encoder_layer = TransformerEncoderLayer(config)
inputs_emb.shape, encoder_layer(inputs_emb).shape

In [ ]:
class Embeddings(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.token_embeddings = nn.Embedding(config.vocab_size,config.hidden_size)
    self.position_embeddings = nn.Embedding(config.max_position_embeddings,config.hidden_size)
    self.layer_norm = nn.LayerNorm(config.hidden_size,eps=1e-12)
    self.dropout = nn.Dropout()

  def forward(self,input_ids):
    seq_length = input_ids.size(1)
    position_ids = torch.arange(seq_length,dtype=torch.long).unsqueeze(0)
    token_embeddings = self.token_embeddings(input_ids)
    position_embeddings = self.position_embeddings(position_ids)
    embeddings = token_embeddings + position_embeddings
    embeddings = self.layer_norm(embeddings)
    embeddings = self.dropout(embeddings)
    return embeddings


In [ ]:
class TransformerEncoder(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.embeddings = Embeddings(config)
    self.layer = nn.ModuleList([TransformerEncoderLayer(config) for _ in range(config.num_hidden_layers)])

  def forward(self,x):
    x = self.embeddings(x)
    for layer in self.layer:
      x = layer(x)
    return x

In [ ]:
class TransformerForSequenceClassificaion(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.encoder = TransformerEncoder(config)
    self.dropout = nn.Dropout(config.hidden_dropout_prob)
    self.classifier = nn.Linear(config.hidden_size,config.num_labels)

  def forward(self,x):
    x = self.encoder(x)[:,0,:]
    x = self.dropout(x)
    x = self.classifier(x)
    return x